# LeetCode 818 — Race Car bằng Python

- Link: https://leetcode.com/problems/race-car/
- Độ khó: **Hard**
- Chủ đề: BFS trên không gian trạng thái `(vị trí, vận tốc)`, Quy hoạch động trên số bit, Memo hoá (top-down), Cắt tỉa trạng thái

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def racecar(self, target: int) -> int:
```

## 1. Giới thiệu & mục tiêu học

Một chiếc xe đua tự lái nằm ở vị trí `0` với vận tốc `+1` trên một trục số vô hạn. Xe chỉ hiểu hai lệnh:

- `A` (Accelerate): `position += speed`, rồi `speed *= 2`.
- `R` (Reverse): nếu `speed > 0` thì `speed = -1`, ngược lại `speed = 1` (vị trí giữ nguyên).

Hỏi chuỗi lệnh **ngắn nhất** để xe dừng đúng ở `target`.

Bài này đẹp vì có **hai góc nhìn hoàn toàn khác nhau**: (1) coi mỗi lệnh là một cạnh trong đồ thị trạng thái `(pos, speed)` rồi chạy BFS; (2) nhận ra cấu trúc nhị phân — chạy liên tục `n` lệnh `A` đi được đúng `2ⁿ − 1` — rồi viết quy hoạch động theo `target`.

Sau notebook này bạn sẽ:

1. Mô hình hoá một bài "chuỗi lệnh ngắn nhất" thành **BFS trên không gian trạng thái**.
2. Biết cách **cắt tỉa** trạng thái (giới hạn vị trí) để BFS chạy nhanh mà vẫn đúng.
3. Hiểu quan sát then chốt: `n` lệnh `A` liên tiếp từ vận tốc 1 đi được `2ⁿ − 1`.
4. Xây dựng **công thức DP** `dp[t]` với hai lựa chọn "vượt qua rồi quay lại" và "dừng trước, lùi một chút rồi đi tiếp".
5. Viết cả bản DP **bottom-up** lẫn **top-down có memo**, và đối chiếu với brute force.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Xe bắt đầu ở `position = 0`, `speed = +1`. Mỗi lệnh:

- `'A'`: `position += speed; speed *= 2`
- `'R'`: `speed = -1 if speed > 0 else 1`

Cho số nguyên dương `target`, trả về **độ dài chuỗi lệnh ngắn nhất** đưa xe tới `position == target` (không quan tâm vận tốc lúc kết thúc).

**Ví dụ 1**

```text
Input: target = 3
Output: 2
Giải thích: "AA": vị trí 0 → 1 → 3.
```

**Ví dụ 2**

```text
Input: target = 6
Output: 5
Giải thích: "AAARA": vị trí 0 → 1 → 3 → 7 → 7 → 6.
```

**Ràng buộc**

- `1 <= target <= 10⁴`

In [1]:
from typing import List

EXAMPLES = [
    (3, 2),
    (6, 5),
    (1, 1),     # "A"
    (2, 4),     # "AARA" hoặc "ARRA"
    (4, 5),
    (5, 7),
    (7, 3),     # "AAA"
]

def simulate(cmds):
    """Chạy một chuỗi lệnh, trả về danh sách (pos, speed) sau mỗi lệnh."""
    pos, speed = 0, 1
    states = [(pos, speed)]
    for c in cmds:
        if c == 'A':
            pos += speed
            speed *= 2
        else:
            speed = -1 if speed > 0 else 1
        states.append((pos, speed))
    return states

print(simulate("AA"))
print(simulate("AAARA"))
for t, exp in EXAMPLES:
    print("target =", t, "-> kỳ vọng", exp)

[(0, 1), (1, 2), (3, 4)]
[(0, 1), (1, 2), (3, 4), (7, 8), (7, -1), (6, -2)]
target = 3 -> kỳ vọng 2
target = 6 -> kỳ vọng 5
target = 1 -> kỳ vọng 1
target = 2 -> kỳ vọng 4
target = 4 -> kỳ vọng 5
target = 5 -> kỳ vọng 7
target = 7 -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1. BFS cho "số bước ít nhất"

Khi mọi bước có cùng chi phí (mỗi lệnh tốn 1), BFS từ trạng thái đầu cho số bước ít nhất tới mọi trạng thái: lần đầu tiên một trạng thái được lấy ra khỏi hàng đợi là khoảng cách nhỏ nhất của nó.

In [2]:
from collections import deque

def bfs_steps(start, goal, neighbors):
    dist = {start: 0}
    q = deque([start])
    while q:
        u = q.popleft()
        if u == goal:
            return dist[u]
        for v in neighbors(u):
            if v not in dist:
                dist[v] = dist[u] + 1
                q.append(v)
    return -1

# ví dụ nhỏ: từ 1 tới 10, mỗi bước được +1 hoặc *2
print(bfs_steps(1, 10, lambda x: [x + 1, x * 2] if x <= 20 else []))

4


### 3.2. Trạng thái là một bộ (tuple)

Trong bài này trạng thái gồm **hai** thông tin `(pos, speed)`. Python cho phép dùng tuple làm khoá của `set`/`dict`, rất tiện cho `visited`.

In [3]:
seen = set()
seen.add((0, 1))
seen.add((1, 2))
print((0, 1) in seen, (0, -1) in seen)

True False


### 3.3. Lũy thừa của 2 và `bit_length`

- `1 << n` bằng `2ⁿ`.
- `t.bit_length()` = số bit cần để biểu diễn `t` = số `n` nhỏ nhất sao cho `2ⁿ > t`, tức `2ⁿ − 1 >= t`.
- `t == (1 << n) - 1` ⇔ `t` có dạng `11...1` (toàn bit 1).

In [4]:
for t in [1, 2, 3, 4, 6, 7, 8, 15, 16]:
    n = t.bit_length()
    print(f"t={t:2}  bin={bin(t):>7}  n={n}  2^n-1={(1 << n) - 1:2}  toàn bit 1? {t == (1 << n) - 1}")

t= 1  bin=    0b1  n=1  2^n-1= 1  toàn bit 1? True
t= 2  bin=   0b10  n=2  2^n-1= 3  toàn bit 1? False
t= 3  bin=   0b11  n=2  2^n-1= 3  toàn bit 1? True
t= 4  bin=  0b100  n=3  2^n-1= 7  toàn bit 1? False
t= 6  bin=  0b110  n=3  2^n-1= 7  toàn bit 1? False
t= 7  bin=  0b111  n=3  2^n-1= 7  toàn bit 1? True
t= 8  bin= 0b1000  n=4  2^n-1=15  toàn bit 1? False
t=15  bin= 0b1111  n=4  2^n-1=15  toàn bit 1? True
t=16  bin=0b10000  n=5  2^n-1=31  toàn bit 1? False


### 3.4. Tổng cấp số nhân

`1 + 2 + 4 + ... + 2ⁿ⁻¹ = 2ⁿ − 1`. Đây chính là quãng đường xe đi được sau `n` lệnh `A` liên tiếp kể từ khi vận tốc là 1.

In [5]:
for n in range(1, 7):
    print(n, sum(2 ** i for i in range(n)), (1 << n) - 1)

1 1 1
2 3 3
3 7 7
4 15 15
5 31 31
6 63 63


### 3.5. Memo hoá với `functools.lru_cache`

Hàm đệ quy có tham số lặp lại → lưu kết quả để không tính lại. `@lru_cache(None)` làm việc đó tự động.

In [6]:
from functools import lru_cache

@lru_cache(None)
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

print(fib(80))

23416728348467685


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Chạy thẳng: các vị trí "đẹp"

Từ `speed = 1`, liên tiếp `A`:

| số lệnh A | vị trí | vận tốc |
|---|---|---|
| 0 | 0 | 1 |
| 1 | 1 | 2 |
| 2 | 3 | 4 |
| 3 | 7 | 8 |
| 4 | 15 | 16 |

→ Ta chỉ dừng "đúng" được ở `1, 3, 7, 15, ...` = `2ⁿ − 1` bằng toàn `A`. Các `target` khác buộc phải có `R`.

### 4.2. Tính tay `target = 6`

- `7 = 2³ − 1` là vị trí đẹp gần nhất ≥ 6. Đi `AAA` tới 7 (3 lệnh), `R` (vận tốc −1), `A` lùi về 6. Tổng `3 + 1 + 1 = 5`.
- Thử dừng trước: `AA` tới 3 (2 lệnh), cần thêm 3 nữa. `RR` (quay đầu hai lần để vận tốc về +1, tốn 2 lệnh) rồi `AA` tới 6: `2 + 2 + 2 = 6` — tệ hơn.

→ Đáp án 5.

### 4.3. Tính tay `target = 4`

- Vượt: `AAA` tới 7, `R`, rồi cần lùi 3 = `2² − 1` → `AA`. Tổng `3 + 1 + 2 = 6`.
- Dừng trước ở 3 (`AA`), `R` (vận tốc −1), lùi `0` bước, `R` (vận tốc +1), đi tiếp 1 → `A`. Tổng `2 + 1 + 0 + 1 + 1 = 5`.
- Dừng trước ở 3, `R`, lùi 1 bước (`A`, về 2), `R`, cần đi thêm 2 → `dp[2] = 4`. Tổng `2 + 1 + 1 + 1 + 4 = 9`.

→ Đáp án 5.

### 4.4. Quan sát: sau mỗi `R`, vận tốc về ±1

Vì `R` luôn đặt vận tốc về `±1`, chuỗi lệnh có dạng `Aᵏ¹ R Aᵏ² R Aᵏ³ ...`. Mỗi khối `Aᵏ` đi được đúng `2ᵏ − 1` theo hướng hiện tại. Bài toán trở thành: chọn các số `k₁, k₂, ...` sao cho tổng có dấu xen kẽ `(2^k₁−1) − (2^k₂−1) + (2^k₃−1) − ... = target`, tối thiểu `Σ kᵢ + (số R)`.

In [7]:
def blocks_to_pos(ks):
    """ks = [k1, k2, ...]: khối A^k1, R, A^k2, R, ... trả về (vị trí, số lệnh)."""
    pos, sign = 0, 1
    for k in ks:
        pos += sign * ((1 << k) - 1)
        sign = -sign
    return pos, sum(ks) + len(ks) - 1

print(blocks_to_pos([3, 1]))        # AAARA -> 6, 5 lệnh
print(blocks_to_pos([2, 0, 1]))     # AARRA -> 4, 5 lệnh
print(simulate("AARRA")[-1])

(6, 5)
(4, 5)
(4, 2)


## 5. Cách 1 — Brute Force: thử mọi chuỗi lệnh theo độ dài tăng dần

**Ý tưởng.** Với `L = 1, 2, 3, ...`, sinh mọi chuỗi gồm `A`/`R` độ dài `L` (`2ᴸ` chuỗi), mô phỏng; chuỗi nào dừng ở `target` thì trả `L`. Vì duyệt `L` tăng dần, lần đầu tìm thấy là ngắn nhất.

In [8]:
from itertools import product

def racecar_brute(target):
    L = 0
    while True:
        L += 1
        for cmds in product("AR", repeat=L):
            pos, speed = 0, 1
            for c in cmds:
                if c == 'A':
                    pos += speed
                    speed *= 2
                else:
                    speed = -1 if speed > 0 else 1
            if pos == target:
                return L

for t, exp in EXAMPLES:
    got = racecar_brute(t)
    print(t, "->", got, "(kỳ vọng", exp, ")")
    assert got == exp

3 -> 2 (kỳ vọng 2 )
6 -> 5 (kỳ vọng 5 )
1 -> 1 (kỳ vọng 1 )
2 -> 4 (kỳ vọng 4 )
4 -> 5 (kỳ vọng 5 )
5 -> 7 (kỳ vọng 7 )
7 -> 3 (kỳ vọng 3 )


**Độ phức tạp.** Nếu đáp án là `L` thì phải thử `2 + 4 + ... + 2ᴸ ≈ 2ᴸ⁺¹` chuỗi, mỗi chuỗi mô phỏng `O(L)`. Với `target = 10⁴`, đáp án khoảng 40–50 → `2⁵⁰` — không thể.

**Nút thắt.** Rất nhiều chuỗi khác nhau dẫn tới **cùng một trạng thái** `(pos, speed)` (ví dụ `RR` không làm gì ngoài đặt vận tốc về +1). Brute force tính lại cùng một trạng thái nhiều lần → nên gộp chúng: đó là BFS với `visited`.

## 6. Bước đột phá

**Insight 1 — Trạng thái chỉ là `(pos, speed)`.** Lịch sử lệnh không quan trọng, chỉ vị trí và vận tốc hiện tại. Vậy bài toán là đường đi ngắn nhất trên đồ thị trạng thái, mỗi trạng thái có đúng 2 cạnh ra (`A` và `R`) → BFS.

**Insight 2 — Cắt tỉa vị trí.** Đi quá xa `target` không bao giờ có lợi: có thể chứng minh chỉ cần xét `0 <= pos <= 2·target` (vượt quá `2·target` thì quãng phải lùi lại còn dài hơn quãng đã đi). Vận tốc luôn là `±2ᵏ` nên chỉ có `O(log target)` giá trị. → Số trạng thái `O(target · log target)`.

**Insight 3 — Cấu trúc nhị phân → DP.** Gọi `n = target.bit_length()`, tức `2ⁿ⁻¹ <= target <= 2ⁿ − 1`. Một lời giải tối ưu có khối `A` đầu tiên dài `n` hoặc `n − 1`:

- *Nếu* `target == 2ⁿ − 1` *thì* đáp án `n` (toàn `A`).
- **Lựa chọn 1 (vượt rồi quay):** đi `n` lệnh `A` tới `2ⁿ − 1` (vượt `target`), `R`, rồi giải bài con ngược chiều với khoảng cách `2ⁿ − 1 − target`: chi phí `n + 1 + dp[2ⁿ − 1 − target]`.
- **Lựa chọn 2 (dừng trước, lùi m, quay lại):** đi `n − 1` lệnh `A` tới `2ⁿ⁻¹ − 1` (chưa tới `target`), `R`, lùi `m` lệnh `A` (`0 <= m < n − 1`) về `2ⁿ⁻¹ − 1 − (2ᵐ − 1)`, `R`, rồi giải bài con khoảng cách `target − 2ⁿ⁻¹ + 2ᵐ`: chi phí `(n − 1) + 1 + m + 1 + dp[target − 2ⁿ⁻¹ + 2ᵐ]`.

Vì bài toán đối xứng (đi trái hay phải như nhau sau khi `R` đưa vận tốc về ±1), `dp[d]` = số lệnh ít nhất để đi được quãng `d` bắt đầu với vận tốc 1 theo hướng cần đi.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — BFS trên trạng thái `(pos, speed)` có cắt tỉa

Mỗi trạng thái sinh hai trạng thái con:

- `A`: `(pos + speed, speed * 2)` — chỉ giữ nếu `0 <= pos + speed <= 2·target`.
- `R`: `(pos, -1 if speed > 0 else 1)`.

In [9]:
def racecar_bfs(target):
    limit = 2 * target
    start = (0, 1)
    seen = {start}
    q = deque([(0, 1, 0)])          # (pos, speed, số lệnh)
    while q:
        pos, speed, d = q.popleft()
        if pos == target:
            return d
        # lệnh A
        np, ns = pos + speed, speed * 2
        if 0 <= np <= limit and (np, ns) not in seen:
            seen.add((np, ns))
            q.append((np, ns, d + 1))
        # lệnh R
        ns = -1 if speed > 0 else 1
        if (pos, ns) not in seen:
            seen.add((pos, ns))
            q.append((pos, ns, d + 1))
    return -1

for t, exp in EXAMPLES:
    assert racecar_bfs(t) == exp
print("BFS có cắt tỉa: OK", [racecar_bfs(t) for t in range(1, 16)])

BFS có cắt tỉa: OK [1, 4, 2, 5, 7, 5, 3, 6, 8, 7, 10, 7, 9, 6, 4]


Có thể kiểm tra **sớm** khi sinh trạng thái con (`np == target` → trả `d + 1`) để tiết kiệm một lớp. Code nộp LeetCode cho cách 2:

In [10]:
class SolutionBFS:
    def racecar(self, target: int) -> int:
        limit = 2 * target
        seen = {(0, 1)}
        q = deque([(0, 1)])
        steps = 0
        while q:
            steps += 1
            for _ in range(len(q)):          # xử lý trọn một lớp
                pos, speed = q.popleft()
                np, ns = pos + speed, speed * 2
                if np == target:
                    return steps
                if 0 <= np <= limit and (np, ns) not in seen:
                    seen.add((np, ns))
                    q.append((np, ns))
                rs = -1 if speed > 0 else 1
                if (pos, rs) not in seen:
                    seen.add((pos, rs))
                    q.append((pos, rs))
        return -1

for t, exp in EXAMPLES:
    assert SolutionBFS().racecar(t) == exp
print("SolutionBFS: OK")

SolutionBFS: OK


**Độ phức tạp cách 2:** số trạng thái `≤ (2·target + 1) · 2 · (log₂(2·target) + 2)` ≈ `O(T log T)`; mỗi trạng thái `O(1)`. Bộ nhớ cùng bậc. Với `T = 10⁴`: vài trăm nghìn trạng thái — chạy được trong Python nhưng chậm hơn DP khá nhiều.

### 7.2. Cách 3 — DP bottom-up theo quãng đường

`dp[t]` = số lệnh ít nhất để đi quãng `t` (bắt đầu `speed = 1`). Tính `dp[1..target]` tăng dần; mỗi `dp[t]` chỉ phụ thuộc các `dp` nhỏ hơn `t`:

- `2ⁿ − 1 − t < t`? Vì `t >= 2ⁿ⁻¹` nên `2ⁿ − 1 − t <= 2ⁿ⁻¹ − 1 < t`. ✔
- `t − 2ⁿ⁻¹ + 2ᵐ < t` vì `m < n − 1` ⇒ `2ᵐ < 2ⁿ⁻¹`. ✔

In [11]:
def racecar_dp(target):
    dp = [0] * (target + 1)
    for t in range(1, target + 1):
        n = t.bit_length()
        full = (1 << n) - 1
        if t == full:
            dp[t] = n
            continue
        # Lựa chọn 1: vượt tới 2^n - 1, quay đầu, đi ngược phần thừa
        best = n + 1 + dp[full - t]
        # Lựa chọn 2: dừng ở 2^(n-1) - 1, quay, lùi m bước, quay, đi tiếp
        for m in range(n - 1):
            back = (1 << m) - 1
            remain = t - ((1 << (n - 1)) - 1) + back
            best = min(best, (n - 1) + 1 + m + 1 + dp[remain])
        dp[t] = best
    return dp[target]

for t, exp in EXAMPLES:
    assert racecar_dp(t) == exp
print("DP bottom-up: OK", [racecar_dp(t) for t in range(1, 16)])

DP bottom-up: OK [1, 4, 2, 5, 7, 5, 3, 6, 8, 7, 10, 7, 9, 6, 4]


Giải thích từng dòng quan trọng:

- `n = t.bit_length()` → `2ⁿ⁻¹ <= t <= 2ⁿ − 1`.
- `full = 2ⁿ − 1` — vị trí đẹp ngay ≥ `t`.
- `best = n + 1 + dp[full - t]`: `n` lệnh `A`, 1 lệnh `R`, rồi phần còn thiếu `full − t` đi theo hướng ngược lại.
- Vòng `m`: `(n − 1)` lệnh `A` tới `2ⁿ⁻¹ − 1`; `R`; `m` lệnh `A` lùi `2ᵐ − 1`; `R`; phần còn lại `remain`.

Code nộp LeetCode cho cách 3:

In [12]:
class SolutionDP:
    def racecar(self, target: int) -> int:
        dp = [0] * (target + 1)
        for t in range(1, target + 1):
            n = t.bit_length()
            if t == (1 << n) - 1:
                dp[t] = n
                continue
            best = n + 1 + dp[(1 << n) - 1 - t]
            for m in range(n - 1):
                best = min(best, n + m + 1 + dp[t - (1 << (n - 1)) + (1 << m)])
            dp[t] = best
        return dp[target]

for t, exp in EXAMPLES:
    assert SolutionDP().racecar(t) == exp
print("SolutionDP: OK")

SolutionDP: OK


Chú ý rút gọn: `(n − 1) + 1 + m + 1 = n + m + 1` và `t − (2ⁿ⁻¹ − 1) + (2ᵐ − 1) = t − 2ⁿ⁻¹ + 2ᵐ`.

**Độ phức tạp cách 3:** `O(T log T)` thời gian (mỗi `t` thử `O(log t)` giá trị `m`), `O(T)` bộ nhớ.

### 7.3. Cách 4 — DP top-down có memo

Cùng công thức, viết đệ quy: chỉ tính những `dp[x]` thực sự cần cho `target`. Thường ít trạng thái hơn bottom-up.

In [13]:
def racecar_memo(target):
    @lru_cache(None)
    def f(t):
        n = t.bit_length()
        if t == (1 << n) - 1:
            return n
        best = n + 1 + f((1 << n) - 1 - t)
        for m in range(n - 1):
            best = min(best, n + m + 1 + f(t - (1 << (n - 1)) + (1 << m)))
        return best
    res = f(target)
    racecar_memo.states = f.cache_info().currsize
    return res

for t, exp in EXAMPLES:
    assert racecar_memo(t) == exp
racecar_memo(10000)
print("Memo OK; số trạng thái thực sự tính cho target=10000:", racecar_memo.states)

Memo OK; số trạng thái thực sự tính cho target=10000: 729


Lưu ý: `f(0)` có `n = 0`, `(1 << 0) - 1 = 0` → trả `0`. Đệ quy sâu tối đa khoảng `log T` tầng cho lựa chọn 1 nhưng lựa chọn 2 có thể giảm chậm; thực tế độ sâu vẫn nhỏ (vài chục) nên không vượt giới hạn đệ quy.

Code nộp LeetCode cho cách 4:

In [14]:
class SolutionMemo:
    def racecar(self, target: int) -> int:
        @lru_cache(None)
        def f(t: int) -> int:
            n = t.bit_length()
            if t == (1 << n) - 1:
                return n
            best = n + 1 + f((1 << n) - 1 - t)
            for m in range(n - 1):
                best = min(best, n + m + 1 + f(t - (1 << (n - 1)) + (1 << m)))
            return best
        return f(target)

for t, exp in EXAMPLES:
    assert SolutionMemo().racecar(t) == exp
print("SolutionMemo: OK")

SolutionMemo: OK


**Độ phức tạp cách 4:** xấu nhất như cách 3 `O(T log T)`, thực tế ít trạng thái hơn.

### 7.4. Cách 5 — Dijkstra trên "khối lệnh"

Một góc nhìn khác: trạng thái là **vị trí** (vận tốc luôn ±1 ngay sau `R`). Từ vị trí `p`, một "cạnh" là khối `Aᵏ R` đi được `2ᵏ − 1` theo hướng hiện tại, giá `k + 1`. Vì chi phí cạnh khác nhau, dùng Dijkstra trên trạng thái `(pos, hướng)`. Khi khối cuối chạm `target` thì không cần `R` cuối.

In [15]:
import heapq

def racecar_dijkstra(target):
    K = target.bit_length() + 1
    limit = 2 * target
    INF = float('inf')
    dist = {(0, 1): 0}
    pq = [(0, 0, 1)]                      # (chi phí, pos, hướng)
    best = INF
    while pq:
        d, pos, direc = heapq.heappop(pq)
        if d >= best:
            break
        if d > dist.get((pos, direc), INF):
            continue
        for k in range(0, K + 1):
            np = pos + direc * ((1 << k) - 1)
            if np == target:
                best = min(best, d + k)       # không cần R cuối
            if np < 0 or np > limit:
                break                         # k lớn hơn chỉ đi xa hơn nữa
            nd = d + k + 1                    # k lệnh A + 1 lệnh R
            state = (np, -direc)
            if nd < dist.get(state, INF):
                dist[state] = nd
                heapq.heappush(pq, (nd, np, -direc))
    return best

for t, exp in EXAMPLES:
    assert racecar_dijkstra(t) == exp, (t, racecar_dijkstra(t))
print("Dijkstra trên khối lệnh: OK")

Dijkstra trên khối lệnh: OK


Vì sao được `break`: với cùng hướng, `k` càng lớn thì `np` càng xa `pos`; một khi `np` đã ra ngoài `[0, 2·target]` thì mọi `k` lớn hơn cũng ở ngoài. Code nộp LeetCode cho cách 5:

In [16]:
class SolutionDijkstra:
    def racecar(self, target: int) -> int:
        K = target.bit_length() + 1
        limit = 2 * target
        dist = {(0, 1): 0}
        pq = [(0, 0, 1)]
        best = float('inf')
        while pq:
            d, pos, direc = heapq.heappop(pq)
            if d >= best:
                break
            if d > dist[(pos, direc)]:
                continue
            for k in range(K + 1):
                np = pos + direc * ((1 << k) - 1)
                if np == target:
                    best = min(best, d + k)
                if np < 0 or np > limit:
                    break
                nd = d + k + 1
                if nd < dist.get((np, -direc), float('inf')):
                    dist[(np, -direc)] = nd
                    heapq.heappush(pq, (nd, np, -direc))
        return best

for t, exp in EXAMPLES:
    assert SolutionDijkstra().racecar(t) == exp
print("SolutionDijkstra: OK")

SolutionDijkstra: OK


**Độ phức tạp cách 5:** `O(T log T)` trạng thái-cạnh, thêm `log` của heap. Chạy được nhưng DP gọn và nhanh hơn. Điểm hay: không cần chứng minh "khối đầu dài `n` hoặc `n−1`" — Dijkstra tự tìm.

## 8. TRACE từng bước

### 8.1. Trace BFS với `target = 6`

In từng lớp BFS (các trạng thái mới phát hiện).

In [17]:
def trace_bfs(target, max_layers=6):
    limit = 2 * target
    seen = {(0, 1)}
    layer = [(0, 1)]
    steps = 0
    while layer and steps < max_layers:
        steps += 1
        nxt = []
        for pos, speed in layer:
            for cmd, (np, ns) in (('A', (pos + speed, speed * 2)), ('R', (pos, -1 if speed > 0 else 1))):
                if cmd == 'A' and not (0 <= np <= limit):
                    continue
                if (np, ns) in seen:
                    continue
                seen.add((np, ns))
                nxt.append((np, ns))
                if np == target:
                    print(f"lớp {steps}: từ {(pos, speed)} --{cmd}--> {(np, ns)}  ĐẾN ĐÍCH sau {steps} lệnh")
                    return steps
        print(f"lớp {steps}: {nxt}")
        layer = nxt
    return None

trace_bfs(6)

lớp 1: [(1, 2), (0, -1)]
lớp 2: [(3, 4), (1, -1)]
lớp 3: [(7, 8), (3, -1), (0, -2), (1, 1)]
lớp 4: [(7, -1), (2, -2), (3, 1), (2, 2)]
lớp 5: từ (7, -1) --A--> (6, -2)  ĐẾN ĐÍCH sau 5 lệnh


5

Bảng trạng thái tóm tắt (chỉ nhánh dẫn tới lời giải):

| lệnh | pos | speed |
|---|---|---|
| (đầu) | 0 | 1 |
| A | 1 | 2 |
| A | 3 | 4 |
| A | 7 | 8 |
| R | 7 | −1 |
| A | 6 | −2 ✔ |

### 8.2. Trace DP bottom-up cho `target = 1..10`

In từng `dp[t]` cùng lựa chọn tốt nhất (lựa chọn 1 hay lựa chọn 2 với `m` nào).

In [18]:
def trace_dp(target):
    dp = [0] * (target + 1)
    for t in range(1, target + 1):
        n = t.bit_length()
        full = (1 << n) - 1
        if t == full:
            dp[t] = n
            print(f"dp[{t:2}] = {n:2}   (t = 2^{n}-1, toàn A)")
            continue
        best, how = n + 1 + dp[full - t], f"vượt tới {full}, R, rồi dp[{full - t}]"
        for m in range(n - 1):
            r = t - (1 << (n - 1)) + (1 << m)
            c = n + m + 1 + dp[r]
            if c < best:
                best, how = c, f"dừng ở {(1 << (n - 1)) - 1}, R, lùi m={m}, R, rồi dp[{r}]"
        dp[t] = best
        print(f"dp[{t:2}] = {best:2}   ({how})")
    return dp[target]

trace_dp(10)

dp[ 1] =  1   (t = 2^1-1, toàn A)
dp[ 2] =  4   (vượt tới 3, R, rồi dp[1])
dp[ 3] =  2   (t = 2^2-1, toàn A)
dp[ 4] =  5   (dừng ở 3, R, lùi m=0, R, rồi dp[1])
dp[ 5] =  7   (dừng ở 3, R, lùi m=1, R, rồi dp[3])
dp[ 6] =  5   (vượt tới 7, R, rồi dp[1])
dp[ 7] =  3   (t = 2^3-1, toàn A)
dp[ 8] =  6   (dừng ở 7, R, lùi m=0, R, rồi dp[1])
dp[ 9] =  8   (dừng ở 7, R, lùi m=1, R, rồi dp[3])
dp[10] =  7   (dừng ở 7, R, lùi m=0, R, rồi dp[3])


7

Đọc bảng: `dp[4] = 5` đến từ "dừng ở 3, R, lùi 0, R, rồi dp[1] = 1": `2 + 1 + 0 + 1 + 1 = 5` — khớp tính tay ở mục 4.3. `dp[5] = 7`: thử lần lượt — dừng ở 3, R, lùi 0, R, rồi `dp[2] = 4` → `2+1+0+1+4 = 8`; vượt tới 7, R, `dp[2] = 4` → `3+1+4 = 8`; dừng ở 3, R, lùi 1 (về 2), R, `dp[3] = 2` → `2+1+1+1+2 = 7` ✔.

In [19]:
# Kiểm tra lại các chuỗi lệnh cụ thể bằng mô phỏng
for cmds, t in [("AARRA", 4), ("AARARAA", 5), ("AAARA", 6)]:
    print(cmds, "->", simulate(cmds)[-1][0], "| độ dài", len(cmds), "| dp =", racecar_dp(t))

AARRA -> 4 | độ dài 5 | dp = 5
AARARAA -> 5 | độ dài 7 | dp = 7
AAARA -> 6 | độ dài 5 | dp = 5


## 9. Code nộp LeetCode

Chọn DP bottom-up (cách 3): ngắn, nhanh (`target = 10⁴` chỉ vài chục ms), không đệ quy.

In [20]:
from typing import List

class Solution:
    def racecar(self, target: int) -> int:
        dp = [0] * (target + 1)
        for t in range(1, target + 1):
            n = t.bit_length()                 # 2^(n-1) <= t <= 2^n - 1
            if t == (1 << n) - 1:              # toàn lệnh A
                dp[t] = n
                continue
            # Vượt tới 2^n - 1, quay đầu, đi ngược phần thừa
            best = n + 1 + dp[(1 << n) - 1 - t]
            # Dừng ở 2^(n-1) - 1, quay, lùi m bước, quay, đi tiếp phần còn lại
            for m in range(n - 1):
                best = min(best, n + m + 1 + dp[t - (1 << (n - 1)) + (1 << m)])
            dp[t] = best
        return dp[target]

for t, exp in EXAMPLES:
    assert Solution().racecar(t) == exp
print("Solution: OK")

Solution: OK


## 10. Vì sao thuật toán đúng

**BFS (cách 2).** Đồ thị trạng thái `(pos, speed)` với hai cạnh trọng số 1. BFS cho đường đi ít cạnh nhất. Phần cần lập luận là **cắt tỉa**:

- *Không cần `pos < 0`:* xe bắt đầu ở 0 và `target > 0`. Một lời giải đi vào vùng âm có thể được "phản chiếu": thay vì quay đầu ở gần 0 rồi chạy âm, ta quay sớm hơn — người ta chứng minh luôn tồn tại lời giải tối ưu không rời đoạn `[0, 2·target]`. (Ta không chứng minh hình thức ở đây mà **kiểm chứng** bằng cách đối chiếu với brute force và DP trên mọi `target` nhỏ.)
- *Không cần `pos > 2·target`:* nếu đã vượt `2·target` thì quãng phải quay về (> `target`) dài hơn cả quãng tới đích từ đầu.

**DP (cách 3, 4).** Mọi chuỗi lệnh có dạng `Aᵏ¹ R Aᵏ² R ...`. Xét khối đầu tiên `Aᵏ¹` với `n = bit_length(t)`:

1. `k₁ >= n + 1`: xe tới `2^k₁ − 1 >= 2ⁿ⁺¹ − 1 > 2t`, phải quay lại hơn `t` — luôn tệ hơn (có thể thay bằng `k₁ = n`). Bỏ qua.
2. `k₁ = n`: xe ở `2ⁿ − 1 >= t`. Nếu bằng `t` → xong (`n` lệnh). Nếu không, phải `R` (1 lệnh) rồi đi quãng `2ⁿ − 1 − t` về phía ngược lại, bắt đầu vận tốc 1 — đúng bài con `dp[2ⁿ − 1 − t]`. → Lựa chọn 1.
3. `k₁ = n − 1`: xe ở `2ⁿ⁻¹ − 1 < t`. Lệnh tiếp phải là `R`. Sau đó khối thứ hai `Aᵐ` đi lùi (`m >= 0`; `m = 0` nghĩa là `RR` liền nhau). Nếu `m >= n − 1` thì lùi về ≤ 0, vô ích. Rồi `R` và bài con quãng còn lại. → Lựa chọn 2 với `0 <= m < n − 1`.
4. `k₁ < n − 1`: có thể chứng minh luôn có lời giải không tệ hơn với `k₁ ∈ {n−1, n}` (đi xa hơn trong khối đầu không tốn thêm `R` và phần còn lại vẫn có thể bù). Đây là điểm tinh tế của lời giải chuẩn; một lần nữa ta **kiểm chứng** bằng brute force và Dijkstra (cách 5 không dùng giả thiết này!).

Bài con đối xứng: sau `R`, vận tốc ±1 và quãng còn lại `d` đi theo hướng ngược — số lệnh tối thiểu chỉ phụ thuộc `d`, không phụ thuộc hướng.

## 11. Phân tích Time/Space

Gọi `T = target`.

| Cách | Time | Space |
|---|---|---|
| 1. Brute force chuỗi lệnh | `O(2ᴸ · L)` với `L` = đáp án | `O(L)` |
| 2. BFS `(pos, speed)` cắt tỉa | `O(T log T)` | `O(T log T)` |
| 3. DP bottom-up | `O(T log T)` | `O(T)` |
| 4. DP top-down memo | `O(T log T)` xấu nhất, thực tế ít hơn | `O(T)` |
| 5. Dijkstra khối lệnh | `O(T log² T)` | `O(T)` |

In [21]:
import time, random

for name, fn in [("BFS", racecar_bfs), ("SolutionBFS", lambda t: SolutionBFS().racecar(t)),
                 ("DP", racecar_dp), ("Memo", racecar_memo),
                 ("Dijkstra", racecar_dijkstra), ("Solution", lambda t: Solution().racecar(t))]:
    t0 = time.perf_counter()
    res = fn(10000)
    print(f"{name:12}: racecar(10000) = {res}  ({(time.perf_counter() - t0) * 1000:.0f} ms)")

BFS         : racecar(10000) = 45  (172 ms)


SolutionBFS : racecar(10000) = 45  (125 ms)
DP          : racecar(10000) = 45  (20 ms)
Memo        : racecar(10000) = 45  (1 ms)


Dijkstra    : racecar(10000) = 45  (119 ms)
Solution    : racecar(10000) = 45  (15 ms)


## 12. Bộ test

In [22]:
methods = {
    "bfs": racecar_bfs,
    "dp": racecar_dp,
    "memo": racecar_memo,
    "dijkstra": racecar_dijkstra,
    "SolutionBFS": lambda t: SolutionBFS().racecar(t),
    "SolutionDP": lambda t: SolutionDP().racecar(t),
    "SolutionMemo": lambda t: SolutionMemo().racecar(t),
    "SolutionDijkstra": lambda t: SolutionDijkstra().racecar(t),
    "Solution": lambda t: Solution().racecar(t),
}

# 12.1 Ví dụ + edge cases
edge = EXAMPLES + [(8, 6), (9, 8), (15, 4), (16, 7), (31, 5), (32, 8)]
for name, fn in methods.items():
    for t, exp in edge:
        assert fn(t) == exp, (name, t, fn(t), exp)
print("Ví dụ + edge cases: OK")

# 12.2 Đối chiếu brute force với mọi target nhỏ (brute chỉ chạy được tới ~20)
for t in range(1, 21):
    ref = racecar_brute(t)
    for name, fn in methods.items():
        assert fn(t) == ref, (name, t)
print("target 1..20: khớp brute force")

# 12.3 Ngẫu nhiên có seed: các cách nhanh khớp nhau (BFS không cắt tỉa giả thiết DP)
rnd = random.Random(818)
for _ in range(60):
    t = rnd.randint(1, 3000)
    vals = {name: fn(t) for name, fn in methods.items()}
    assert len(set(vals.values())) == 1, (t, vals)
print("60 target ngẫu nhiên: mọi cách khớp nhau")

# 12.4 Toàn bộ 1..10^4: DP so với Dijkstra (hai cách dựa trên lập luận khác nhau)
d = [0] * 10001          # bảng dp đầy đủ 1..10^4 (tính một lần)
for t in range(1, 10001):
    n = t.bit_length()
    if t == (1 << n) - 1:
        d[t] = n
        continue
    b = n + 1 + d[(1 << n) - 1 - t]
    for m in range(n - 1):
        b = min(b, n + m + 1 + d[t - (1 << (n - 1)) + (1 << m)])
    d[t] = b
for t in rnd.sample(range(1, 10001), 40):
    assert d[t] == racecar_dijkstra(t) == racecar_bfs(t), t
print("Mẫu 40 target trong 1..10^4: DP = Dijkstra = BFS")

Ví dụ + edge cases: OK
target 1..20: khớp brute force


60 target ngẫu nhiên: mọi cách khớp nhau


Mẫu 40 target trong 1..10^4: DP = Dijkstra = BFS


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Cắt tỉa quá chặt: cấm `pos > target`

**Code sai:**

In [23]:
def wrong_prune(target):
    seen = {(0, 1)}
    q = deque([(0, 1, 0)])
    while q:
        pos, speed, d = q.popleft()
        if pos == target:
            return d
        np, ns = pos + speed, speed * 2
        if 0 <= np <= target and (np, ns) not in seen:     # SAI: không cho vượt target
            seen.add((np, ns)); q.append((np, ns, d + 1))
        rs = -1 if speed > 0 else 1
        if (pos, rs) not in seen:
            seen.add((pos, rs)); q.append((pos, rs, d + 1))
    return -1

print("target=6  sai:", wrong_prune(6), " đúng:", Solution().racecar(6))

target=6  sai: 6  đúng: 5


**Vì sao sai:** lời giải tối ưu cho 6 là `AAARA` — phải **vượt** tới 7 rồi lùi. Cấm vượt buộc xe đi đường vòng dài hơn.

**Code đúng:** cho phép tới `2·target`.

### Lỗi 2 — Đánh dấu `visited` chỉ theo `pos` (quên vận tốc)

**Code sai:**

In [24]:
def wrong_visited_pos(target):
    seen = {0}
    q = deque([(0, 1, 0)])
    while q:
        pos, speed, d = q.popleft()
        if pos == target:
            return d
        for np, ns in ((pos + speed, speed * 2), (pos, -1 if speed > 0 else 1)):
            if 0 <= np <= 2 * target and np not in seen:   # SAI: chỉ theo vị trí
                seen.add(np); q.append((np, ns, d + 1))
    return -1

print("target=5  sai:", wrong_visited_pos(5), " đúng:", Solution().racecar(5))

target=5  sai: -1  đúng: 7


**Vì sao sai:** cùng một vị trí nhưng vận tốc khác nhau là hai tình huống khác hẳn. Lệnh `R` không đổi vị trí nên bị loại ngay → không bao giờ quay đầu được.

**Code đúng:** khoá `visited` là `(pos, speed)`.

### Lỗi 3 — DP quên lựa chọn 2 (chỉ xét "vượt rồi quay")

**Code sai:**

In [25]:
def wrong_dp_only_overshoot(target):
    dp = [0] * (target + 1)
    for t in range(1, target + 1):
        n = t.bit_length()
        if t == (1 << n) - 1:
            dp[t] = n
        else:
            dp[t] = n + 1 + dp[(1 << n) - 1 - t]          # SAI: thiếu lựa chọn dừng trước
    return dp[target]

for t in [4, 5, 9]:
    print(f"target={t}  sai:", wrong_dp_only_overshoot(t), " đúng:", Solution().racecar(t))

target=4  sai: 6  đúng: 5
target=5  sai: 8  đúng: 7
target=9  sai: 10  đúng: 8


**Vì sao sai:** với `target = 4`, vượt tới 7 rồi lùi 3 tốn 6 lệnh; dừng ở 3 rồi `RRA` chỉ tốn 5.

**Code đúng:** thêm vòng `for m in range(n - 1)`.

### Lỗi 4 — Sai biên vòng `m` (cho `m = n − 1`)

Với `m = n − 1`, xe lùi `2ⁿ⁻¹ − 1` về đúng vị trí 0 → bài con `dp[t]` chính nó (chưa tính, bằng 0) → kết quả nhỏ sai.

In [26]:
def wrong_m_range(target):
    dp = [0] * (target + 1)
    for t in range(1, target + 1):
        n = t.bit_length()
        if t == (1 << n) - 1:
            dp[t] = n; continue
        best = n + 1 + dp[(1 << n) - 1 - t]
        for m in range(n):                                  # SAI: m tới n-1
            best = min(best, n + m + 1 + dp[t - (1 << (n - 1)) + (1 << m)])
        dp[t] = best
    return dp[target]

print("target=5  sai:", wrong_m_range(5), " đúng:", Solution().racecar(5))

target=5  sai: 6  đúng: 7


**Vì sao sai:** với `m = n − 1`, phần còn lại là `t − 2ⁿ⁻¹ + 2ⁿ⁻¹ = t` → `dp[t]` tham chiếu chính nó (đang là 0), cho chi phí giả `2n`. Với `target = 5` (`n = 3`) ra `6` thay vì `7`. **Code đúng:** `range(n - 1)`.

## 14. Mẹo & nhận dạng pattern

- **"Chuỗi lệnh / thao tác ngắn nhất để biến trạng thái đầu thành trạng thái đích"** → BFS trên không gian trạng thái. Hỏi ngay: trạng thái gồm những gì? Có hữu hạn không? Cắt tỉa thế nào?
- **Thao tác nhân đôi / lũy thừa 2** → nghĩ tới `bit_length`, dạng `2ⁿ − 1`, DP theo số bit.
- **"Vượt rồi quay lại" vs "dừng trước rồi bù"** là khuôn mẫu chung của nhiều bài DP trên trục số (ví dụ 991 Broken Calculator, 397 Integer Replacement).
- Khi BFS có trạng thái vô hạn, luôn tìm **chặn trên tự nhiên** (ở đây `2·target`) và kiểm chứng bằng brute force.
- Bài liên quan: **752 Open the Lock**, **773 Sliding Puzzle**, **991 Broken Calculator**, **1654 Minimum Jumps to Reach Home** (BFS với chặn vị trí).

## 15. Biến thể

1. **In chuỗi lệnh tối ưu**: BFS lưu `parent` của mỗi trạng thái rồi truy ngược.
2. **Yêu cầu dừng với vận tốc dương** (hoặc vận tốc ±1): thêm điều kiện vào trạng thái đích.
3. **`target` có thể âm**: do đối xứng, `racecar(-t)` = số lệnh để đi `t` sau một lệnh `R` đầu tiên… cẩn thận: xe ban đầu hướng dương, nên `racecar(−t) = 1 + dp[t]` hoặc tốt hơn qua đường khác — BFS trên đoạn `[−2t, 2t]` xử lý tổng quát.
4. **Lệnh `A` có hệ số khác (ví dụ `speed *= 3`)**: DP theo cơ số 3 hoặc BFS tổng quát.
5. **Nhiều truy vấn `target`**: tính bảng `dp[1..10⁴]` một lần, trả lời `O(1)` mỗi truy vấn.

Minh hoạ biến thể 1 — khôi phục chuỗi lệnh:

In [27]:
def racecar_path(target):
    limit = 2 * target
    parent = {(0, 1): None}
    q = deque([(0, 1)])
    while q:
        pos, speed = q.popleft()
        if pos == target:
            cmds, cur = [], (pos, speed)
            while parent[cur] is not None:
                prev, c = parent[cur]
                cmds.append(c)
                cur = prev
            return ''.join(reversed(cmds))
        for c, st in (('A', (pos + speed, speed * 2)), ('R', (pos, -1 if speed > 0 else 1))):
            if c == 'A' and not (0 <= st[0] <= limit):
                continue
            if st not in parent:
                parent[st] = ((pos, speed), c)
                q.append(st)
    return None

for t in [3, 4, 5, 6, 9, 100]:
    p = racecar_path(t)
    assert simulate(p)[-1][0] == t and len(p) == Solution().racecar(t)
    print(t, p)

3 AA
4 AARRA
5 AARARAA
6 AAARA
9 AAARARAA
100 AAAAAAARAAAAARAARRA


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | thử mọi chuỗi theo độ dài | `O(2ᴸ L)` | `O(L)` | đối chiếu, `target ≤ 20` |
| 2. BFS cắt tỉa | đồ thị `(pos, speed)`, `0 ≤ pos ≤ 2T` | `O(T log T)` | `O(T log T)` | dễ nghĩ ra, dễ in đường đi |
| 3. DP bottom-up | khối đầu dài `n` hoặc `n−1` | `O(T log T)` | `O(T)` | **nộp bài**, nhanh nhất |
| 4. DP memo | như 3, chỉ tính trạng thái cần | `O(T log T)` | `O(T)` | khi chỉ cần một `target` |
| 5. Dijkstra khối lệnh | cạnh = `AᵏR`, giá `k+1` | `O(T log² T)` | `O(T)` | không cần chứng minh cấu trúc |

## 17. Bài tập

**Bài 1 (đoán output).** Chuỗi `"AARAAA"` đưa xe tới vị trí nào? `"ARRAA"`?

**Bài 2 (tính tay).** Tính `dp[9]` bằng công thức, ghi rõ lựa chọn tốt nhất.

**Bài 3 (điền TODO).** Hoàn thiện `racecar_neg(target)` cho phép `target` âm hoặc dương (BFS trên `[−2|T|, 2|T|]`).

**Bài 4 (tự viết lại).** Viết lại DP top-down **không** dùng `lru_cache` (dùng dict thủ công).

**Bài 5 (mini challenge).** Biến thể: yêu cầu xe dừng ở `target` **với vận tốc dương**. Viết BFS và so sánh với đáp án gốc cho `target = 1..30`.

In [28]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def racecar_neg_todo(target):
    # TODO: limit = 2*abs(target); trạng thái (pos, speed); cho phép -limit <= pos <= limit
    pass

# Bài 4 — TODO
def racecar_memo_dict_todo(target):
    # TODO: memo = {}; def f(t): if t in memo: ...
    pass

# Bài 5 — TODO
def racecar_positive_todo(target):
    # TODO: BFS, đích là (target, speed) với speed > 0
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: dùng `simulate`.
- Mức 2: `AA` → 3, `R` → vận tốc −1, `AAA` lùi 1+2+4 = 7.
- Mức 3: `−4`; chuỗi thứ hai: `A`(1) `RR`(vận tốc +1) `AA`(+3) → 4.

**Bài 2.**
- Mức 1: `n = 4`, `full = 15`.
- Mức 2: lựa chọn 1: `4 + 1 + dp[6] = 10`. Lựa chọn 2 với `m = 0, 1, 2`: `dp[2], dp[3], dp[5]`.
- Mức 3: `m=0`: `4+0+1+dp[2]=9`; `m=1`: `4+1+1+dp[3]=8`; `m=2`: `4+2+1+dp[5]=14` → `8`.

**Bài 3.**
- Mức 1: giống BFS cách 2.
- Mức 2: đổi điều kiện cắt tỉa sang `abs(np) <= limit`.
- Mức 3: `target = −1` → `RA` = 2.

**Bài 4.**
- Mức 1: `memo = {0: 0}`.
- Mức 2: tra `memo` trước khi tính.
- Mức 3: so sánh với `Solution` cho `1..200`.

**Bài 5.**
- Mức 1: chỉ thay điều kiện dừng.
- Mức 2: nếu chạm `target` với vận tốc âm, có thể thêm `R` (1 lệnh) — vậy đáp án ≤ gốc + 1.
- Mức 3: BFS trực tiếp vẫn là cách an toàn nhất.

## 19. Đáp án tham khảo

In [29]:
# Bài 1
assert simulate("AARAAA")[-1][0] == -4
assert simulate("ARRAA")[-1][0] == 4

# Bài 2
assert racecar_dp(9) == 8

# Bài 3
def racecar_neg(target):
    if target == 0:
        return 0
    limit = 2 * abs(target)
    seen = {(0, 1)}
    q = deque([(0, 1, 0)])
    while q:
        pos, speed, d = q.popleft()
        if pos == target:
            return d
        for np, ns in ((pos + speed, speed * 2), (pos, -1 if speed > 0 else 1)):
            if abs(np) <= limit and (np, ns) not in seen:
                seen.add((np, ns)); q.append((np, ns, d + 1))
    return -1

assert racecar_neg(-1) == 2
assert all(racecar_neg(t) == Solution().racecar(t) for t in range(1, 60))
print("racecar_neg(-1..-6):", [racecar_neg(-t) for t in range(1, 7)])

# Bài 4
def racecar_memo_dict(target):
    memo = {0: 0}
    def f(t):
        if t in memo:
            return memo[t]
        n = t.bit_length()
        if t == (1 << n) - 1:
            memo[t] = n
            return n
        best = n + 1 + f((1 << n) - 1 - t)
        for m in range(n - 1):
            best = min(best, n + m + 1 + f(t - (1 << (n - 1)) + (1 << m)))
        memo[t] = best
        return best
    return f(target)

assert all(racecar_memo_dict(t) == Solution().racecar(t) for t in range(1, 300))

# Bài 5
def racecar_positive(target):
    limit = 2 * target
    seen = {(0, 1)}
    q = deque([(0, 1, 0)])
    while q:
        pos, speed, d = q.popleft()
        if pos == target and speed > 0:
            return d
        for np, ns in ((pos + speed, speed * 2), (pos, -1 if speed > 0 else 1)):
            if 0 <= np <= limit and (np, ns) not in seen:
                seen.add((np, ns)); q.append((np, ns, d + 1))
    return -1

for t in range(1, 31):
    a, b = Solution().racecar(t), racecar_positive(t)
    assert a <= b <= a + 1
print("vận tốc dương:", [racecar_positive(t) for t in range(1, 11)])
print("Tất cả đáp án đúng")

racecar_neg(-1..-6): [2, 4, 3, 6, 7, 5]
vận tốc dương: [1, 4, 2, 5, 7, 6, 3, 6, 8, 7]
Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template DP:**

```python
dp = [0] * (target + 1)
for t in range(1, target + 1):
    n = t.bit_length()
    if t == (1 << n) - 1: dp[t] = n; continue
    best = n + 1 + dp[(1 << n) - 1 - t]                     # vượt rồi quay
    for m in range(n - 1):                                   # dừng trước, lùi m, quay
        best = min(best, n + m + 1 + dp[t - (1 << (n - 1)) + (1 << m)])
    dp[t] = best
return dp[target]
```

**Template BFS trạng thái:**

```python
seen = {(0, 1)}; q = deque([(0, 1, 0)])
while q:
    pos, sp, d = q.popleft()
    if pos == target: return d
    for np, ns in ((pos + sp, sp * 2), (pos, -1 if sp > 0 else 1)):
        if 0 <= np <= 2 * target and (np, ns) not in seen:
            seen.add((np, ns)); q.append((np, ns, d + 1))
```

**Checklist trước khi submit:**

- [ ] `visited` theo **cặp** `(pos, speed)`, không chỉ `pos`.
- [ ] Cho phép **vượt** `target` (tới `2·target`).
- [ ] DP có **cả hai** lựa chọn; vòng `m` là `range(n - 1)`.
- [ ] `n = t.bit_length()`; trường hợp `t == 2ⁿ − 1` trả `n`.
- [ ] Thử nhanh `target = 3 → 2`, `6 → 5`, `4 → 5`.